# DistilBERT evaluation on external captions

Part of the code for *Danger Detection through Audio Captioning and Classification Fusion* (Electronics, MDPI). Paper section: 4.5.

Applies the saved DistilBERT models to the CoNeTTE captions of the external datasets.

**Environment.** Written for Google Colab (GPU or CPU, Hugging Face transformers); see `environment/`. Paths point to a Google Drive layout. Edit the path cell to your own copy of the datasets. Random seed: 42.

**Inputs.** Fine-tuned DistilBERT models and the external caption files.

**Outputs.** Per-segment DistilBERT predictions and metrics.


# DistilBERT Evaluation on XD-Violence

This notebook evaluates the saved DistilBERT `best_model` folders from the low-threshold experiments on the CoNeTTE captions of XD-Violence.

It saves per-model predictions, per-dataset metrics, and one combined comparison table.

Important: DistilBERT evaluates text, so use the caption CSVs, not the audio-only CNN manifests.


## 1. Install Dependencies

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas

## 2. Imports

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 3. Mount Drive and Configure Paths

Edit these paths if your Drive layout is different.

`MODEL_ROOT` should be the folder that contains experiment folders such as `low_threshold__random__standard_ce/best_model`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
MODEL_ROOT = Path(
    '/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/DIstilBERT/training/low_threshold_experiments'
)

XDVIODET_CSV = Path(
    '/content/drive/MyDrive/Colab Notebooks/FINALS/Conette_Captions/xdviodet_conette_clotho_captions_clean.csv'
)

OUTPUT_DIR = MODEL_ROOT / 'external_eval_xdviodet_tut'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('MODEL_ROOT =', MODEL_ROOT)
print('XDVIODET_CSV =', XDVIODET_CSV)
print('OUTPUT_DIR =', OUTPUT_DIR)

## 4. Settings

In [ ]:
MAX_LENGTH = 128
BATCH_SIZE = 32
THRESHOLD = 0.50

LABEL_MAP = {'no_danger_risk': 0, 'danger_risk': 1}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

DATASETS = {
    'xdviodet': {
        'path': XDVIODET_CSV,
        'caption_col': 'caption',
        'label_col': 'y_true',
        'id_col': 'target_audio_name',
        'group_col': 'bag_id',
    },
}

## 5. Discover Best Models

In [ ]:
def discover_best_models(model_root: Path) -> pd.DataFrame:
    rows = []
    for model_dir in sorted(model_root.glob('low_threshold__*/best_model')):
        if not (model_dir / 'config.json').exists():
            continue
        experiment_dir = model_dir.parent
        metrics_path = experiment_dir / 'metrics.json'
        metrics = {}
        if metrics_path.exists():
            with metrics_path.open('r', encoding='utf-8') as f:
                metrics = json.load(f)
        rows.append({
            'experiment_name': experiment_dir.name,
            'model_dir': str(model_dir),
            'val_recall_danger_1': metrics.get('eval_recall_danger_1'),
            'val_f1_danger_1': metrics.get('eval_f1_danger_1'),
            'val_macro_f1': metrics.get('eval_macro_f1'),
            'val_accuracy': metrics.get('eval_accuracy'),
        })
    model_df = pd.DataFrame(rows)
    if model_df.empty:
        raise FileNotFoundError(
            f'No best_model folders found under {model_root}. '
            'Check MODEL_ROOT or finish the training notebook first.'
        )
    return model_df.sort_values(
        ['val_recall_danger_1', 'val_f1_danger_1', 'val_macro_f1'],
        ascending=[False, False, False],
        na_position='last',
    ).reset_index(drop=True)

model_df = discover_best_models(MODEL_ROOT)
model_df

## 6. Load External Datasets

In [ ]:
def normalize_risk_label(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower().replace('_', ' ')
    if text in {'1', 'danger', 'danger risk', 'danger_risk', 'risk'}:
        return 1
    if text in {'0', 'no danger', 'no danger risk', 'no_danger_risk', 'normal'}:
        return 0
    return np.nan


def load_eval_dataset(name: str, spec: dict) -> pd.DataFrame:
    path = spec['path']
    if not path.exists():
        raise FileNotFoundError(f'Missing {name} CSV: {path}')
    df = pd.read_csv(path)

    caption_col = spec['caption_col']
    if caption_col not in df.columns:
        raise ValueError(f'{name} is missing caption column: {caption_col}')

    df = df.copy()
    df['caption'] = df[caption_col].astype(str).str.strip()
    df = df[df['caption'] != ''].copy()

    label_col = spec.get('label_col')
    if label_col and label_col in df.columns:
        df['y_true'] = pd.to_numeric(df[label_col], errors='coerce')
    elif 'risk_class' in df.columns:
        df['y_true'] = df['risk_class'].map(normalize_risk_label)
    elif 'risk_label' in df.columns:
        df['y_true'] = df['risk_label'].map(normalize_risk_label)
    else:
        df['y_true'] = np.nan

    df['y_true'] = pd.to_numeric(df['y_true'], errors='coerce')
    df = df.dropna(subset=['y_true']).copy()
    df['y_true'] = df['y_true'].astype(int)
    df['semantic_label_final'] = df['y_true'].map(INV_LABEL_MAP)
    df['dataset_name'] = name

    id_col = spec.get('id_col')
    if id_col and id_col in df.columns:
        df['eval_id'] = df[id_col].astype(str)
    else:
        df['eval_id'] = np.arange(len(df)).astype(str)

    group_col = spec.get('group_col')
    if group_col and group_col in df.columns:
        df['eval_group'] = df[group_col].astype(str)
    else:
        df['eval_group'] = df['eval_id']

    print(f'\n{name}: rows={len(df)}')
    print(df['semantic_label_final'].value_counts().to_string())
    return df.reset_index(drop=True)

eval_datasets = {name: load_eval_dataset(name, spec) for name, spec in DATASETS.items()}

## 7. Prediction and Metrics Helpers

In [ ]:
def predict_texts(model_dir: Path, captions: list[str]) -> tuple[np.ndarray, np.ndarray]:
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    all_probs = []
    with torch.no_grad():
        for start in range(0, len(captions), BATCH_SIZE):
            batch = captions[start:start + BATCH_SIZE]
            enc = tokenizer(
                batch,
                truncation=True,
                padding=True,
                max_length=MAX_LENGTH,
                return_tensors='pt',
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
            all_probs.append(probs)

    danger_probs = np.concatenate(all_probs) if all_probs else np.array([])
    y_pred = (danger_probs >= THRESHOLD).astype(int)
    return danger_probs, y_pred


def compute_binary_metrics(y_true, danger_probs, y_pred) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    danger_probs = np.asarray(danger_probs).astype(float)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], zero_division=0
    )

    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_not_danger_0': precision[0],
        'recall_not_danger_0': recall[0],
        'f1_not_danger_0': f1[0],
        'support_not_danger_0': int(support[0]),
        'precision_danger_1': precision[1],
        'recall_danger_1': recall[1],
        'f1_danger_1': f1[1],
        'support_danger_1': int(support[1]),
        'macro_f1': float((f1[0] + f1[1]) / 2),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
        'threshold': THRESHOLD,
    }

    if len(np.unique(y_true)) == 2:
        metrics['roc_auc'] = roc_auc_score(y_true, danger_probs)
        metrics['average_precision'] = average_precision_score(y_true, danger_probs)
    else:
        metrics['roc_auc'] = np.nan
        metrics['average_precision'] = np.nan

    metrics['false_positive_rate'] = (metrics['fp'] / (metrics['fp'] + metrics['tn'])) if (metrics['fp'] + metrics['tn']) else np.nan
    metrics['false_negative_rate'] = (metrics['fn'] / (metrics['fn'] + metrics['tp'])) if (metrics['fn'] + metrics['tp']) else np.nan
    return metrics


def aggregate_by_group(pred_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for group, group_df in pred_df.groupby('eval_group'):
        y_true_values = group_df['y_true'].dropna().astype(int).unique()
        y_true = int(max(y_true_values)) if len(y_true_values) else np.nan
        max_prob = float(group_df['distilbert_danger_prob'].max())
        mean_prob = float(group_df['distilbert_danger_prob'].mean())
        rows.append({
            'eval_group': group,
            'n_segments': len(group_df),
            'y_true': y_true,
            'max_danger_prob': max_prob,
            'mean_danger_prob': mean_prob,
            'y_pred_max': int(max_prob >= THRESHOLD),
            'y_pred_mean': int(mean_prob >= THRESHOLD),
        })
    return pd.DataFrame(rows)

## 8. Run External Evaluation

In [ ]:
summary_rows = []
group_summary_rows = []

for _, model_row in model_df.iterrows():
    experiment_name = model_row['experiment_name']
    model_dir = Path(model_row['model_dir'])
    print(f'\n{"=" * 80}')
    print('MODEL:', experiment_name)
    print('DIR:', model_dir)

    for dataset_name, eval_df in eval_datasets.items():
        print(f'  evaluating {dataset_name} ({len(eval_df)} rows)')
        danger_probs, y_pred = predict_texts(model_dir, eval_df['caption'].tolist())

        pred_df = eval_df.copy()
        pred_df['distilbert_danger_prob'] = danger_probs
        pred_df['distilbert_prediction'] = y_pred
        pred_df['distilbert_prediction_name'] = pred_df['distilbert_prediction'].map(INV_LABEL_MAP)

        model_output_dir = OUTPUT_DIR / experiment_name
        model_output_dir.mkdir(parents=True, exist_ok=True)
        pred_path = model_output_dir / f'{dataset_name}_predictions.csv'
        pred_df.to_csv(pred_path, index=False)

        metrics = compute_binary_metrics(
            pred_df['y_true'].values,
            pred_df['distilbert_danger_prob'].values,
            pred_df['distilbert_prediction'].values,
        )
        metrics.update({
            'experiment_name': experiment_name,
            'dataset_name': dataset_name,
            'model_dir': str(model_dir),
            'predictions_path': str(pred_path),
        })
        summary_rows.append(metrics)

        group_df = aggregate_by_group(pred_df)
        group_path = model_output_dir / f'{dataset_name}_group_predictions.csv'
        group_df.to_csv(group_path, index=False)

        if group_df['y_true'].nunique(dropna=True) > 0:
            group_metrics = compute_binary_metrics(
                group_df['y_true'].values,
                group_df['max_danger_prob'].values,
                group_df['y_pred_max'].values,
            )
            group_metrics.update({
                'experiment_name': experiment_name,
                'dataset_name': dataset_name,
                'aggregation': 'group_max',
                'group_predictions_path': str(group_path),
            })
            group_summary_rows.append(group_metrics)

        print(
            f"    acc={metrics['accuracy']:.4f} | "
            f"danger_recall={metrics['recall_danger_1']:.4f} | "
            f"danger_f1={metrics['f1_danger_1']:.4f} | "
            f"FP={metrics['fp']} FN={metrics['fn']}"
        )

summary_df = pd.DataFrame(summary_rows)
group_summary_df = pd.DataFrame(group_summary_rows)

summary_path = OUTPUT_DIR / 'distilbert_external_eval_summary.csv'
group_summary_path = OUTPUT_DIR / 'distilbert_external_eval_group_summary.csv'
summary_df.to_csv(summary_path, index=False)
group_summary_df.to_csv(group_summary_path, index=False)

print('\nsaved:', summary_path)
print('saved:', group_summary_path)

## 9. Compare Segment-Level Results

In [ ]:
display_cols = [
    'dataset_name',
    'experiment_name',
    'accuracy',
    'precision_danger_1',
    'recall_danger_1',
    'f1_danger_1',
    'macro_f1',
    'roc_auc',
    'average_precision',
    'tn',
    'fp',
    'fn',
    'tp',
    'false_positive_rate',
    'false_negative_rate',
]
summary_df.sort_values(
    ['dataset_name', 'recall_danger_1', 'f1_danger_1', 'macro_f1'],
    ascending=[True, False, False, False],
)[display_cols]

## 10. Compare Group-Level Results

Group-level metrics use max danger probability over all segments for the same source item.

In [ ]:
if not group_summary_df.empty:
    display(group_summary_df.sort_values(
        ['dataset_name', 'recall_danger_1', 'f1_danger_1', 'macro_f1'],
        ascending=[True, False, False, False],
    )[[
        'dataset_name',
        'experiment_name',
        'aggregation',
        'accuracy',
        'precision_danger_1',
        'recall_danger_1',
        'f1_danger_1',
        'macro_f1',
        'roc_auc',
        'average_precision',
        'tn',
        'fp',
        'fn',
        'tp',
        'false_positive_rate',
        'false_negative_rate',
    ]])
else:
    print('No group-level summary available.')

## 11. Best Model Per Dataset

In [ ]:
ranked = summary_df.copy()
ranked['meets_recall_threshold'] = ranked['recall_danger_1'] >= 0.80
ranked = ranked.sort_values(
    ['dataset_name', 'meets_recall_threshold', 'f1_danger_1', 'macro_f1', 'recall_danger_1'],
    ascending=[True, False, False, False, False],
)

best_per_dataset = ranked.groupby('dataset_name', as_index=False).head(1)
best_per_dataset[display_cols + ['predictions_path']]